# Interactive ERH Analysis

This notebook allows interactive exploration of simulation results using Plotly and ipywidgets.
It demonstrates how to load batch results and visualize key metrics.

In [ ]:
import os
import sys
import json
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from ipywidgets import interact, widgets

# Ensure project root is in path
project_root = os.path.abspath(os.path.join(os.getcwd(), '..', '..'))
if project_root not in sys.path:
    sys.path.insert(0, project_root)

from simulation.core.output_writer import save_json_result

## Load Data
Point this to your directory containing `sim_result_*.json` files.

In [ ]:
def load_results(directory):
    data = []
    if not os.path.exists(directory):
        print(f"Directory {directory} not found.")
        return pd.DataFrame()
        
    for f in os.listdir(directory):
        if f.endswith(".json") and f.startswith("sim_result"):
            path = os.path.join(directory, f)
            try:
                with open(path, 'r') as fh:
                    res = json.load(fh)
                    row = {
                        'filename': f,
                        'dist': res['config']['complexity_dist'],
                        'N': res['config']['num_actions'],
                        'mistake_rate': res['metrics']['mistake_rate'],
                        'alpha': res['metrics'].get('estimated_exponent')
                    }
                    data.append(row)
            except Exception as e:
                print(f"Skipping {f}: {e}")
    return pd.DataFrame(data)

# Default path (adjust as needed)
results_dir = os.path.join(project_root, 'simulation_results')
df = load_results(results_dir)
print(f"Loaded {len(df)} experiments.")
df.head()

## Interactive Visualization

In [ ]:
@interact
def plot_metrics(metric=['mistake_rate', 'alpha'], split_by=['dist', 'N']):
    if df.empty:
        print("No data loaded.")
        return
        
    fig = px.box(df, x=split_by, y=metric, color=split_by, points="all",
                 title=f"{metric} by {split_by}")
    
    if metric == 'alpha':
        fig.add_hline(y=0.5, line_dash="dash", line_color="red", annotation_text="ERH Limit")
        
    fig.show()